In [ ]:
# Pyomo Process Optimization Example

#This notebook demonstrates how to build and solve a nonlinear optimization problem using **Pyomo** and **IPOPT**.

#All data here are randomly generated for demonstration purposes only.


import numpy as np
from pyomo.environ import (
    ConcreteModel, Var, Objective, ConstraintList,
    SolverFactory, log10, minimize, value
)
import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# -----------------------------
# Generate dummy parameters
# -----------------------------
np.random.seed(42)
param_list = np.random.rand(5, 16, 4)   # 5 parameter sets (each 16x4)
r_values = np.random.rand(5, 4)         # 5 target vectors (each 4-length)

def solve_system_pyomo(param, r):
    """Solve a 4-variable nonlinear optimization system using Pyomo."""
    model = ConcreteModel()

    # Decision variables
    model.a = Var(bounds=(-9, 0))
    model.b = Var(bounds=(-8, 0))
    model.c = Var(bounds=(-7, 0))
    model.d = Var(bounds=(-9, 0))

    def calc_l_and_sum(m):
        vars_ = [10 ** m.a, 10 ** m.b, 10 ** m.c, 10 ** m.d]
        logs_ = [log10(v) + off for v, off in zip(vars_, [9, 8, 7, 9])]
        return vars_, logs_, sum(vars_)

    # Constraints
    model.constraints = ConstraintList()
    for i, idx_group in enumerate([[0,1,2,3],[4,5,6,7],[8,9,10,11],[12,13,14,15]]):
        vars_, logs_, s = calc_l_and_sum(model)
        eq = sum(
            (param[idx][0] * logs_[j]**param[idx][3] /
            (param[idx][1]**param[idx][3] + logs_[j]**param[idx][3]) +
            param[idx][2]) * vars_[j]
            for j, idx in enumerate(idx_group)
        ) / s - r[i]
        model.constraints.add(eq == 0)

    # Objective (minimize squared residuals)
    model.obj = Objective(expr=sum(con.body**2 for con in model.constraints.values()), sense=minimize)

    solver = SolverFactory("ipopt")
    solver.options["tol"] = 1e-8

    try:
        solver.solve(model, tee=False)
        vars_, _, _ = calc_l_and_sum(model)
        solution = [value(v) for v in vars_]
        return solution, value(model.obj)
    except Exception:
        return [None]*4, None


# -----------------------------
# Run optimization on dummy data
# -----------------------------
if __name__ == "__main__":
    for i, param in enumerate(param_list):
        r = r_values[np.random.randint(0, len(r_values))]
        solution, error = solve_system_pyomo(param, r)
        print(f"Case {i+1}: solution = {solution}, error = {error:.3e}")
